**Silver to Gold: Building BI Ready Tables**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, DoubleType, DateType, TimestampType
from pyspark.sql import Row

In [0]:
catalog_name = "ecommerce"

In [0]:
df_products = spark.read.table(f"{catalog_name}.silver.slv_products")
df_brands = spark.read.table(f"{catalog_name}.silver.slv_brands")
df_category = spark.read.table(f"{catalog_name}.silver.slv_category")


In [0]:
df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

In [0]:
display(spark.sql("select  * from v_products limit 5"))
display(spark.sql("select distinct brand_code, brand_name from v_brands"))
display(spark.sql("select distinct trim(category_code), trim(category_name) from v_category"))

In [0]:
# make sure we're in the correct catalog
spark.sql(f"USE CATALOG {catalog_name}")
          

In [0]:
%sql

-- Build brandsxcategory mapping and write Gold table
CREATE OR REPLACE TABLE gold.gld_dim_products AS

WITH brands_categories AS (
SELECT 
    b.brand_code, 
    b.brand_name, 
    c.category_code, 
    c.category_name 
FROM v_brands b 
INNER JOIN v_category c 
ON b.category_code = c.category_code
)
SELECT 
    p.product_id, 
    p.sku,
    p.brand_code, 
    COALESCE(bc.brand_name, 'Not Available') AS brand_name,
    p.category_code, 
    COALESCE(bc.category_name, 'Not Available') AS category_name,
    p.color, 
    p.size, 
    p.material, 
    p.weight_grams, 
    p.length_cm, 
    p.width_cm, 
    p.height_cm, 
    p.rating_count, 
    p.file_name, 
    p.ingest_timestamp
FROM v_products p
LEFT JOIN brands_categories bc 
ON p.brand_code = bc.brand_code